## 1. 필요한 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# QSAR 전처리 관련
from qsar_preprocess import QSARPreprocessor
from standardize_smiles import standardize_smiles
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen

# Jupyter 파일 업로드 위젯
from IPython.display import display, HTML, FileLink
import ipywidgets as widgets
from ipywidgets import FileUpload, Output, VBox, HBox, Label, Button

print("✓ 모든 라이브러리 임포트 완료")
print(f"  실행 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Excel 파일 업로드 및 데이터 로딩

In [ ]:
# 간단한 파일 업로드 (로컬 Jupyter용)
from pathlib import Path
import os

print("=" * 70)
print("📁 엑셀 파일 업로드 및 로딩")
print("=" * 70)

# test_data 폴더와 현재 디렉토리의 Excel 파일 목록 표시
test_folder = Path('test_data')
excel_files = (
    list(Path('.').glob('*.xlsx')) + 
    list(Path('.').glob('*.xls')) + 
    list(Path('.').glob('*.csv')) +
    list(test_folder.glob('*.xlsx')) +
    list(test_folder.glob('*.xls')) +
    list(test_folder.glob('*.csv'))
)

if excel_files:
    print("\n📋 사용 가능한 파일 목록:")
    for i, file in enumerate(excel_files, 1):
        print(f"  {i}. {file}")
else:
    print("\n⚠️  현재 디렉토리와 test_data 폴더에 Excel/CSV 파일이 없습니다.")

print("\n" + "=" * 70)
print("아래 셀에서 파일 경로를 입력하여 로드하세요.")
print("(예: test_data/test_molecules.xlsx 또는 test_molecules.xlsx)")
print("=" * 70)

# 파일명 입력 위젯
file_input = widgets.Text(
    value='test_data/test_molecules.xlsx',
    placeholder='파일 경로 입력 (예: test_data/test_molecules.xlsx)',
    description='파일 경로:',
    style={'description_width': '100px'}
)

output_area = Output()

def load_file(b=None):
    """파일 로드 및 SMILES 컬럼 확인"""
    global df_original, df_processed
    
    with output_area:
        output_area.clear_output(wait=True)
        
        filename = file_input.value.strip()
        
        if not filename:
            print("❌ 파일 경로를 입력하세요.")
            return
        
        # 파일 존재 확인
        if not os.path.exists(filename):
            print(f"❌ 파일을 찾을 수 없습니다: {filename}")
            print(f"\n현재 디렉토리: {os.getcwd()}")
            print("\n사용 가능한 파일:")
            for file in excel_files:
                print(f"  - {file}")
            return
        
        # 파일 읽기
        try:
            if filename.lower().endswith(('.xlsx', '.xls')):
                df_original = pd.read_excel(filename)
            else:
                df_original = pd.read_csv(filename)
            
            print(f"✅ 파일 로드 성공: {filename}")
            print(f"  행 수: {len(df_original)}")
            print(f"  열 수: {len(df_original.columns)}")
            print(f"  열 이름: {list(df_original.columns)}")
            print(f"\n📊 데이터 미리보기 (처음 5행):")
            display(df_original.head())
            
        except Exception as e:
            print(f"❌ 파일 로드 실패: {e}")
            import traceback
            traceback.print_exc()

# 로드 버튼
load_button = Button(
    description='파일 로드',
    button_style='success',
    tooltip='Click to load the file'
)
load_button.on_click(load_file)

# UI 표시
display(HBox([file_input, load_button]))
display(output_area)


## 2-1. 테스트 데이터 생성 (선택사항)

실제 Excel 파일이 없다면, 아래 셀을 실행하여 테스트 데이터를 생성할 수 있습니다.

In [ ]:
# 테스트 데이터 생성 (선택)
# 실제 파일이 없을 때 사용하기 위한 샘플 데이터
from pathlib import Path

# test_data 폴더 경로 인식
test_folder = Path('test_data')
test_folder.mkdir(exist_ok=True)

test_data = {
    'Name': [
        'Aspirin', 'Benzene', 'Caffeine', 'Ibuprofen', 'Acetate',
        'Ethanol', 'Paracetamol', 'Invalid', 'Warfarin', 'Theophylline'
    ],
    'SMILES': [
        'CC(=O)Oc1ccccc1C(=O)O',              # Aspirin
        'c1ccccc1',                            # Benzene
        'CN1C=NC2=C1C(=O)N(C(=O)N2C)C.[Cl-]',# Caffeine (with Cl)
        'CC(C)Cc1ccc(cc1)C(C)C(=O)O',         # Ibuprofen
        'CC(=O)[O-].[Na+]',                   # Acetate salt
        'CCO',                                 # Ethanol
        'CC(=O)Nc1ccc(O)cc1',                 # Paracetamol
        'INVALID_SMILES_XYZ',                 # Invalid
        'CC(=O)CC(C)C1=C(O)c2ccccc2OC1=O',   # Warfarin
        'CN1c2ccccc2C(=O)N(C1=O)C'            # Theophylline
    ]
}

df_test = pd.DataFrame(test_data)

# 테스트 폴더에 파일 저장
test_filename = test_folder / 'test_molecules.xlsx'
df_test.to_excel(test_filename, index=False)

print(f'✓ 테스트 데이터 생성: {test_filename}')
print(f'  저장 위치: {test_folder.absolute()}')
print(f'  분자 수: {len(df_test)}')
print(f'\n샘플 데이터:')
print(df_test.head())
print(f'\n위의 파일 업로드 위젯에서 test_data/{test_filename.name}를 선택하세요.')


## 3. 전처리 실행 함수

In [ ]:
def run_preprocessing(df, smiles_col, use_molvs=True, remove_salts=True, 
                     filter_organics=True, protomer_norm=True):
    """
    QSAR 전처리 파이프라인 실행
    
    Returns:
        tuple: (df_result, stats_dict)
    """
    preprocessor = QSARPreprocessor(
        use_molvs=use_molvs,
        remove_salts=remove_salts,
        filter_organics=filter_organics,
        verbose=False
    )
    
    # 전처리 실행
    df_result = preprocessor.preprocess_dataframe(
        df.copy(),
        smiles_column=smiles_col,
        keep_original=True,
        drop_invalid=True
    )
    
    # 통계 수집
    stats = {
        'input_count': preprocessor.stats['input_count'],
        'valid_smiles': preprocessor.stats['valid_smiles'],
        'output_count': preprocessor.stats['output_count'],
        'invalid_smiles': preprocessor.stats['invalid_smiles'],
        'failed_processing': preprocessor.stats['failed_processing'],
        'pass_rate': (preprocessor.stats['output_count'] / preprocessor.stats['input_count'] * 100) if preprocessor.stats['input_count'] > 0 else 0,
    }
    
    return df_result, stats

print("✓ 전처리 함수 정의 완료")

## 4. 통계 분석 및 시각화

In [ ]:
def print_preprocessing_stats(df_orig, df_proc, stats, smiles_col):
    """
    전처리 통계 출력
    """
    print("\n" + "="*70)
    print("QSAR 전처리 파이프라인 - 최종 통계")
    print("="*70)
    
    # 1. 기본 통계
    print("\n[1] 기본 처리 통계")
    print("-" * 70)
    print(f"입력 분자 수:         {stats['input_count']:6d}")
    print(f"유효한 SMILES:        {stats['valid_smiles']:6d} ({stats['valid_smiles']/stats['input_count']*100:.1f}%)")
    print(f"처리 완료 분자:       {stats['output_count']:6d}")
    print(f"\n총 Pass Rate:        {stats['pass_rate']:6.1f}%")
    print("-" * 70)
    
    # 2. 제거된 분자 분석
    print("\n[2] 제거된 분자 분석")
    print("-" * 70)
    removed = stats['invalid_smiles'] + stats['failed_processing']
    print(f"잘못된 SMILES:        {stats['invalid_smiles']:6d}")
    print(f"처리 중 제거:         {stats['failed_processing']:6d}")
    print(f"총 제거된 분자:       {removed:6d} ({removed/stats['input_count']*100:.1f}%)")
    print("-" * 70)
    
    # 3. 분자량 분석
    print("\n[3] 분자량 분포 (전처리 후)")
    print("-" * 70)
    if len(df_proc) > 0 and 'SMILES_clean' in df_proc.columns:
        mws = []
        for smi in df_proc['SMILES_clean'].dropna():
            try:
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    mws.append(Descriptors.ExactMolWt(mol))
            except:
                pass
        
        if mws:
            mws = np.array(mws)
            print(f"평균 분자량:          {np.mean(mws):8.2f} g/mol")
            print(f"중앙값:               {np.median(mws):8.2f} g/mol")
            print(f"범위:                 {np.min(mws):8.2f} - {np.max(mws):8.2f} g/mol")
            print(f"표준편차:             {np.std(mws):8.2f} g/mol")
    print("-" * 70)
    
    # 4. 처리 설정
    print("\n[4] 처리 설정")
    print("-" * 70)
    print(f"MolVS 표준화:         O")
    print(f"염 제거:              O")
    print(f"pH 7.4 프로토머:      O")
    print(f"유기물 필터링:        O")
    print("-" * 70)
    print(f"처리 완료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70 + "\n")

print("✓ 통계 출력 함수 정의 완료")

## 5. 시각화 함수

In [ ]:
def visualize_preprocessing(df_orig, df_proc, stats, smiles_col):
    """
    전처리 결과 시각화
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('QSAR Preprocessing Pipeline - Visualization Dashboard', fontsize=16, fontweight='bold')
    
    # 1. Pass Rate 파이 차트
    ax = axes[0, 0]
    sizes = [stats['output_count'], stats['invalid_smiles'] + stats['failed_processing']]
    labels = [f"Pass ({stats['pass_rate']:.1f}%)", f"Fail ({100-stats['pass_rate']:.1f}%)"]
    colors = ['#2ecc71', '#e74c3c']
    ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax.set_title('Pass Rate', fontweight='bold')
    
    # 2. 제거 이유 분석
    ax = axes[0, 1]
    removal_reasons = [
        ('Invalid SMILES', stats['invalid_smiles']),
        ('Processing Error', stats['failed_processing'])
    ]
    reasons, counts = zip(*removal_reasons)
    ax.bar(reasons, counts, color=['#e74c3c', '#f39c12'])
    ax.set_ylabel('Count')
    ax.set_title('Removal Analysis', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # 3. 분자량 분포
    ax = axes[1, 0]
    if len(df_proc) > 0 and 'SMILES_clean' in df_proc.columns:
        mws = []
        for smi in df_proc['SMILES_clean'].dropna():
            try:
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    mws.append(Descriptors.ExactMolWt(mol))
            except:
                pass
        
        if mws:
            ax.hist(mws, bins=30, color='#3498db', edgecolor='black', alpha=0.7)
            ax.axvline(np.mean(mws), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(mws):.0f}')
            ax.set_xlabel('Molecular Weight (g/mol)')
            ax.set_ylabel('Frequency')
            ax.set_title('Molecular Weight Distribution', fontweight='bold')
            ax.legend()
            ax.grid(alpha=0.3)
    
    # 4. 처리 단계별 통계
    ax = axes[1, 1]
    stages = ['Input', 'Valid\nSMILES', 'Output']
    stage_counts = [stats['input_count'], stats['valid_smiles'], stats['output_count']]
    colors_stages = ['#95a5a6', '#3498db', '#2ecc71']
    ax.bar(stages, stage_counts, color=colors_stages, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Count')
    ax.set_title('Processing Stages', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # 값 레이블 추가
    for i, (stage, count) in enumerate(zip(stages, stage_counts)):
        ax.text(i, count + 5, str(count), ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    print("\n✓ Visualization complete")

print("✓ Visualization function defined")

## 6. 전체 처리 워크플로우 (위의 버튼 클릭 시 실행됨)

In [ ]:
## 전처리 실행 함수
print("\n" + "=" * 70)
print("🚀 전처리 시작")
print("=" * 70)

smiles_column_input = widgets.Text(
    value='SMILES',
    placeholder='SMILES column name',
    description='SMILES Column:',
    style={'description_width': '100px'}
)

preprocess_output = Output()

def run_full_preprocessing(b=None):
    """완전한 전처리 파이프라인 실행"""
    global df_original, df_processed
    
    with preprocess_output:
        preprocess_output.clear_output(wait=True)
        
        # df_original 확인
        if df_original is None:
            print("❌ 먼저 파일을 로드하세요.")
            return
        
        # SMILES 열 확인
        smiles_col = smiles_column_input.value.strip()
        if not smiles_col:
            print("❌ SMILES 컬럼명을 입력하세요.")
            return
        
        if smiles_col not in df_original.columns:
            print(f"❌ '{smiles_col}' 열을 찾을 수 없습니다.")
            print(f"사용 가능한 열: {list(df_original.columns)}")
            return
        
        print(f"✅ SMILES 열 확인: '{smiles_col}'")
        print(f"\n🔄 전처리 진행 중...\n")
        
        try:
            # preprocessed_data 폴더 생성
            output_dir = Path('preprocessed_data')
            output_dir.mkdir(exist_ok=True)
            
            # 전처리 실행
            df_processed, stats = run_preprocessing(df_original, smiles_col)
            
            # 통계 출력
            print_preprocessing_stats(df_original, df_processed, stats, smiles_col)
            
            # 시각화
            visualize_preprocessing(df_original, df_processed, stats, smiles_col)
            
            # 결과 미리보기
            print("\n[7] 전처리 결과 샘플 (처음 10개):")
            print("-" * 70)
            if len(df_processed) > 0:
                display_cols = [col for col in [smiles_col, 'SMILES_clean'] if col in df_processed.columns]
                display(df_processed[display_cols].head(10))
            else:
                print("처리된 분자가 없습니다.")
            
            # 결과 저장 (preprocessed_data 폴더)
            output_filename = output_dir / f"preprocessed_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            df_processed.to_csv(output_filename, index=False)
            print(f"\n✅ 결과 저장: {output_filename}")
            
        except Exception as e:
            print(f"❌ 전처리 중 오류: {e}")
            import traceback
            traceback.print_exc()

# 전처리 버튼
preprocess_button = Button(
    description='전처리 실행',
    button_style='info',
    tooltip='Click to start preprocessing'
)
preprocess_button.on_click(run_full_preprocessing)

# UI 표시
print("\n입력 데이터 로드 후, SMILES 컬럼명을 입력하고 '전처리 실행' 버튼을 클릭하세요.\n")
display(HBox([smiles_column_input, preprocess_button]))
display(preprocess_output)

print("✓ 전처리 모듈 준비 완료")


## 7. 빠른 시작 가이드 (Quick Start)

In [ ]:
print('=' * 80)
print(' ' * 15 + '🚀 QSAR PREPROCESSING PIPELINE - QUICK START GUIDE')
print('=' * 80)
print()
print('📋 3-Stage Workflow with 400+ Molecular Descriptors')
print()
print('┌─ STAGE 1: FILE LOADING ─────────────────────────────────────────────────┐')
print('│ [Cell 2] Load File                                                      │')
print('│  • Supports: Excel (.xlsx, .xls), CSV formats                           │')
print('│  • Input: test_data/ folder (default) or custom path                    │')
print('│  • Output: df_original (DataFrame preview)                              │')
print('│  • Action: Enter filename → Click "Load File" button                    │')
print('└─────────────────────────────────────────────────────────────────────────┘')
print()
print('┌─ OPTIONAL: GENERATE TEST DATA ──────────────────────────────────────────┐')
print('│ [Cell 3] Test Data Generation                                           │')
print('│  • Creates: 10 sample molecules in test_data/                           │')
print('│  • File: test_molecules.xlsx                                            │')
print('│  • Use case: Quick testing before using real data                       │')
print('└─────────────────────────────────────────────────────────────────────────┘')
print()
print('┌─ STAGE 2: SMILES PREPROCESSING ─────────────────────────────────────────┐')
print('│ [Cells 4-7] Preprocessing Pipeline                                      │')
print('│  • MolVS standardization: Enabled                                       │')
print('│  • Salt removal: Enabled (TOXBAI SMARTS patterns)                       │')
print('│  • pH 7.4 protomer normalization: Enabled                               │')
print('│  • Organic molecule filtering: Enabled                                  │')
print('│  • Stereochemistry: Preserved                                           │')
print('│  • Output: preprocessed_data/preprocessed_YYYYMMDD_HHMMSS.csv          │')
print('│  • Action: Enter SMILES column name → Click "Run Preprocessing"         │')
print('│  • Dashboard: 4-panel statistics & visualization                        │')
print('└─────────────────────────────────────────────────────────────────────────┘')
print()
print('┌─ STAGE 3: MOLECULAR DESCRIPTORS (400+) ────────────────────────────────┐')
print('│ [Cells 10-11] Descriptor Calculation                                    │')
print('│  • Total Descriptors: 400+ comprehensive molecular properties           │')
print('│  • Categories:                                                          │')
print('│    - AUTOCORR2D: 192 descriptors (2D spatial autocorrelation)           │')
print('│    - Functional Groups: 60+ (fr_aldehyde, fr_ketone, etc.)              │')
print('│    - Basic: 80+ (MolWt, LogP, TPSA, NumHDonors, etc.)                  │')
print('│    - VSA: 40+ (PEOE_VSA, EState_VSA, etc.)                              │')
print('│    - Chi/Kappa: 20 (Chi0, Chi1, Kappa1, Kappa2, etc.)                   │')
print('│    - BCUT2D: 8 eigenvalue descriptors                                   │')
print('│  • Output: molecular_descriptors/descriptors_YYYYMMDD_HHMMSS.csv       │')
print('│  • Action: Click "Calculate Descriptors" button                         │')
print('│  • Note: Requires completed Stage 2 preprocessing                       │')
print('└─────────────────────────────────────────────────────────────────────────┘')
print()
print('📂 AUTO-CREATED OUTPUT DIRECTORIES:')
print('  • test_data/ ................. Input molecules (Excel/CSV)')
print('  • preprocessed_data/ ......... Stage 2 output (cleaned SMILES)')
print('  • molecular_descriptors/ ..... Stage 3 output (400+ features)')
print()
print('⚙️  ALL FILES ARE TIMESTAMPED (YYYYMMDD_HHMMSS) FOR VERSION CONTROL')
print('=' * 80)


## 8. 분자 설명자(Molecular Descriptors) 계산


In [ ]:
# 분자 설명자 계산 함수 정의
# RDKit의 모든 가능한 설명자 (402개 + 추가 계산 설명자)

from rdkit.Chem import Descriptors, AllChem, Crippen
from rdkit import Chem

descriptor_funcs = {}

# ===== RDKit 모든 Descriptors 모듈 (402개) =====
# 각 설명자를 동적으로 추가
for name in dir(Descriptors):
    if not name.startswith('_'):
        obj = getattr(Descriptors, name)
        if callable(obj) and not name.startswith('_'):
            try:
                # 테스트 분자로 검증
                test_mol = Chem.MolFromSmiles('CCO')
                if test_mol and callable(obj):
                    result = obj(test_mol)
                    if isinstance(result, (int, float)):
                        descriptor_funcs[name] = obj
            except:
                pass

# ===== 추가 계산 설명자 (분자 구조에서 직접 계산) =====
def get_num_atoms(mol):
    """전체 원자 개수"""
    return mol.GetNumAtoms()

def get_num_bonds(mol):
    """전체 결합 개수"""
    return len(mol.GetBonds())

def get_heavy_atoms_alt(mol):
    """무거운 원자 개수 (대안)"""
    return Descriptors.HeavyAtomCount(mol)

descriptor_funcs['NumAtoms'] = get_num_atoms
descriptor_funcs['NumBonds_Alt'] = get_num_bonds
descriptor_funcs['HeavyAtoms_Alt'] = get_heavy_atoms_alt

def calc_descriptors(smiles: str) -> dict:
    """
    SMILES로부터 모든 분자 설명자 계산 (402+ 개)
    
    Args:
        smiles: Input SMILES string
    
    Returns:
        Dictionary with descriptor values
    """
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return {name: None for name in descriptor_funcs}
    
    values = {}
    for name, func in descriptor_funcs.items():
        try:
            values[name] = func(mol)
        except:
            values[name] = None
    
    return values

print("=" * 70)
print("📊 분자 설명자 계산 모듈 - 최대 전체 확장 (400+ 개 설명자)")
print("=" * 70)
print(f"\n✓ 총 {len(descriptor_funcs)}개 설명자 계산 가능")
print("\n포함된 설명자 카테고리:")
print("  [AUTOCORR2D] 192개 - 2D 자기상관 설명자")
print("  [BCUT2D] 8개 - 경계 조건 고유값")
print("  [Chi] 12개 - 연결성 지수")
print("  [EState_VSA] 11개 - 정전기 위치 표면적")
print("  [PEOE_VSA] 14개 - 부분 전하 위치 표면적")
print("  [SMR_VSA] 9개 - 몰 굴절률 위치 표면적")
print("  [SlogP_VSA] 12개 - LogP 위치 표면적")
print("  [VSA_EState] 9개 - VSA 기반 EState")
print("  [기능기] 60개+ - 약물 대사 관련 기능기")
print("  [기본 설명자] 80개+ - 분자 무게, 로그P, TPSA 등")
print("  [기타] 추가 계산 설명자")
print(f"\n✓ 총 {len(descriptor_funcs)}개 설명자 계산 함수 정의 완료")
print("\n주요 설명자 예시:")
print("  - 분자 무게 및 크기")
print("  - 친지방성 (LogP, MR)")
print("  - 극성 (TPSA, 극성 표면적)")
print("  - 수소결합 (HBA, HBD)")
print("  - 위상학적 특성 (Chi, Kappa, 고리)")
print("  - 표면적 기반 설명자 (VSA, PEOE)")
print("  - 자동상관 설명자 (192개)")
print("  - 약물 기능기 (60개+)")


In [ ]:
# 설명자 계산 실행
print("\n" + "=" * 70)
print("🔬 분자 설명자 계산 실행")
print("=" * 70)

descriptor_output = Output()

def run_descriptor_calculation(b=None):
    """전처리된 데이터로부터 분자 설명자 계산"""
    global df_processed
    
    with descriptor_output:
        descriptor_output.clear_output(wait=True)
        
        # df_processed 확인
        if df_processed is None or len(df_processed) == 0:
            print("❌ 먼저 전처리를 실행하세요.")
            return
        
        if 'SMILES_clean' not in df_processed.columns:
            print("❌ 전처리된 SMILES 데이터(SMILES_clean)가 없습니다.")
            return
        
        print("🔄 분자 설명자 계산 중...\n")
        
        try:
            # molecular_descriptors 폴더 생성
            descriptor_dir = Path('molecular_descriptors')
            descriptor_dir.mkdir(exist_ok=True)
            
            # 설명자 계산
            df_descriptors = df_processed.copy()
            descriptor_list = []
            
            total = len(df_descriptors)
            for idx, smiles in enumerate(df_descriptors['SMILES_clean'], 1):
                desc_dict = calc_descriptors(smiles)
                descriptor_list.append(desc_dict)
                
                # 진행 상황 표시 (매 10개마다)
                if idx % max(1, total // 10) == 0 or idx == total:
                    print(f"  진행률: {idx}/{total} ({100*idx/total:.1f}%)")
            
            # DataFrame으로 변환
            df_desc = pd.DataFrame(descriptor_list)
            df_descriptors = pd.concat([df_descriptors, df_desc], axis=1)
            
            print(f"\n✅ 설명자 계산 완료")
            print(f"  계산된 설명자 수: {len(descriptor_funcs)}")
            print(f"  처리된 분자 수: {len(df_descriptors)}")
            
            # 통계 출력
            print("\n[설명자 통계]")
            print("-" * 70)
            print(df_descriptors[list(descriptor_funcs.keys())].describe().round(2))
            print("-" * 70)
            
            # 결과 미리보기
            print("\n[설명자 계산 결과 샘플 (처음 5개)]:")
            print("-" * 70)
            display(df_descriptors[['SMILES_clean'] + list(descriptor_funcs.keys())].head())
            
            # 결과 저장
            output_filename = descriptor_dir / f"descriptors_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            df_descriptors.to_csv(output_filename, index=False)
            print(f"\n✅ 결과 저장: {output_filename}")
            
        except Exception as e:
            print(f"❌ 설명자 계산 중 오류: {e}")
            import traceback
            traceback.print_exc()

# 설명자 계산 버튼
descriptor_button = Button(
    description='설명자 계산',
    button_style='warning',
    tooltip='Click to calculate molecular descriptors'
)
descriptor_button.on_click(run_descriptor_calculation)

# UI 표시
print("\n전처리 완료 후, '설명자 계산' 버튼을 클릭하세요.\n")
display(descriptor_button)
display(descriptor_output)

print("✓ 설명자 계산 모듈 준비 완료")
